# GloGEM vs GloGEMflow: Alpine Ensemble

Compares two geometry-evolution approaches across 12 representative Alpine glaciers
spanning a 400-fold area range (0.21–…81.84 km²) and all major Alpine sub-ranges.

| Model | Geometry approach |
|-------|-------------------|
| **GloGEM Δh** | Δh parameterisation (Huss et al. 2010) |
| **GloGEMflow** | SIA flowline dynamics (Zekollari et al. 2019) |

Both models use identical calibrated MB parameters from the Hugonnet 2000–2020 calibration.  
Climate forcing: BCC-CSM2-MR / ssp126.

**Sections**
1. Configuration and data loading
2. Ensemble relative-volume timeseries
3. Per-glacier area + volume panels
4. 2100 volume-loss summary
5. MB bias table (Hugonnet period)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import rcParams
from matplotlib.ticker import MaxNLocator

try:
    from cmcrameri import cm as cmc
    COL_DH   = cmc.roma(0.92)
    COL_FLOW  = cmc.roma(0.08)
except ImportError:
    print('cmcrameri not found — using fallback colours')
    COL_DH   = '#1565C0'
    COL_FLOW  = '#B71C1C'

COL_HUG    = '0.35'
ALPHA_BAND  = 0.15
ALPHA_SHADE = 0.18

rcParams['font.family']       = 'sans-serif'
rcParams['font.sans-serif']   = ['Liberation Sans', 'Arial', 'DejaVu Sans']
rcParams['axes.unicode_minus'] = False
rcParams['figure.dpi']        = 200
rcParams['axes.spines.top']   = False
rcParams['axes.spines.right'] = False
rcParams['axes.labelsize']    = 11
rcParams['legend.framealpha'] = 0.9
rcParams['legend.fontsize']   = 9
rcParams['axes.grid']         = True
rcParams['grid.alpha']        = 0.35
rcParams['grid.linestyle']    = '--'
rcParams['grid.linewidth']    = 0.5

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────────
_base = '/scratch_net/vierzack04_fourth/jabeer/GloGEM/glogemflow_development'
_sub  = 'monthly/CentralEurope/files/files_original/BCC-CSM2-MR/ssp126'
DIR_DH   = f'{_base}/ensemble_dhdt/{_sub}/'
DIR_FLOW  = f'{_base}/ensemble_flow/{_sub}/'
DIR_HUG   = '/home/jabeer/projects/glogemflow_development/GloGEM/test/data/geodetic/RGIv7.0/aggregated_2000_2020/'

RUN_SUFFIX = '_r1_GloGEMflow_ensemble'

# ── Time axis ──────────────────────────────────────────────────────────────────
TRAN0   = 1940
N_YEARS = 161
years   = np.arange(TRAN0, TRAN0 + N_YEARS)

HUG_PERIOD = (2000, 2019)
i_hug0     = HUG_PERIOD[0] - TRAN0
i_hug1     = HUG_PERIOD[1] - TRAN0 + 1
years_hug  = years[i_hug0:i_hug1]

# ── Ensemble ───────────────────────────────────────────────────────────────────
GLACIER_IDS = [
    '02596', '01225', '00757', '02611',
    '03664', '03239', '02216', '00978',
    '03135', '00804', '01238', '02072',
]
GLACIER_NAMES = [
    'Aletsch',      'Gorner area',   'Mer de Glace', 'Fiescher area',
    'Pasterze',     'Ortler',        'Morteratsch',  'Valais-M',
    'Oetztal',      'Silvretta',     'Valais-S',     'Plaine Morte',
]
GLACIER_AREAS = np.array([
    81.84, 56.47, 30.67, 31.23,
    17.74, 16.89, 15.76, 15.43,
     9.28,  1.95,  1.56,  0.21,
])
N_GL = len(GLACIER_IDS)

print(f'Ensemble: {N_GL} glaciers, {GLACIER_AREAS.min():.2f}–{GLACIER_AREAS.max():.2f} km²')
print(f'Δh param. dir: {DIR_DH}')
print(f'flow dir:      {DIR_FLOW}')

In [3]:
def read_glogem_row(filepath, glacier_id):
    """Return annual timeseries for one glacier from a GloGEM .dat file."""
    with open(filepath) as f:
        f.readline()
        for line in f:
            parts = line.split()
            if parts and parts[0] == str(glacier_id):
                return np.array(parts[1:], dtype=float)
    raise ValueError(f'Glacier {glacier_id} not found in {filepath}')


def read_hugonnet_mb(directory, glacier_id):
    """Return (MB, err) in m w.e. yr-1 from Hugonnet 2000-2020 file."""
    filepath = directory + '11_mb_glspec.dat'
    with open(filepath) as f:
        for _ in range(3):
            f.readline()
        for line in f:
            parts = line.split()
            if len(parts) < 6:
                continue
            gid = parts[0][9:14]
            if gid == str(glacier_id):
                return float(parts[4]), float(parts[5])
    raise ValueError(f'Glacier {glacier_id} not found in Hugonnet file')


print('Helper functions defined.')

Helper functions defined.


In [ ]:
mb_dh     = np.array([read_glogem_row(DIR_DH + f'centraleurope_Annual_Balance_sfc{RUN_SUFFIX}.dat', g) for g in GLACIER_IDS])
mb_flow   = np.array([read_glogem_row(DIR_FLOW  + f'centraleurope_Annual_Balance_sfc{RUN_SUFFIX}.dat', g) for g in GLACIER_IDS])
area_dh   = np.array([read_glogem_row(DIR_DH + f'centraleurope_Area{RUN_SUFFIX}.dat', g)             for g in GLACIER_IDS])
area_flow = np.array([read_glogem_row(DIR_FLOW  + f'centraleurope_Area{RUN_SUFFIX}.dat', g)             for g in GLACIER_IDS])
vol_dh    = np.array([read_glogem_row(DIR_DH + f'centraleurope_Volume{RUN_SUFFIX}.dat', g)           for g in GLACIER_IDS])
vol_flow  = np.array([read_glogem_row(DIR_FLOW  + f'centraleurope_Volume{RUN_SUFFIX}.dat', g)           for g in GLACIER_IDS])
ela_dh    = np.array([read_glogem_row(DIR_DH + f'centraleurope_ELA{RUN_SUFFIX}.dat', g)              for g in GLACIER_IDS])
ela_flow  = np.array([read_glogem_row(DIR_FLOW  + f'centraleurope_ELA{RUN_SUFFIX}.dat', g)              for g in GLACIER_IDS])

hug_mb, hug_err = zip(*[read_hugonnet_mb(DIR_HUG, g) for g in GLACIER_IDS])
hug_mb  = np.array(hug_mb)
hug_err = np.array(hug_err)

print(f'Loaded {N_GL} glaciers x {N_YEARS} years x 4 variables for both models.')

In [ ]:
# ── Alps-wide batch run: load the same 12 glaciers from the batch .dat files ───
_alps_base = '/scratch_net/vierzack04_fourth/jabeer/GloGEM/glogemflow_development/alps_flow'
_alps_sub  = 'monthly/CentralEurope/files/files_original/BCC-CSM2-MR/ssp126'

# Which batch file each glacier lives in
ALPS_BATCH = {
    '00757': 3,  '00804': 4,  '00978': 5,
    '01225': 6,  '01238': 6,  '02596': 7,
    '02611': 8,  '02072': 11, '02216': 11,
    '03239': 13, '03135': 13, '03664': 16,
}

def alps_dat(var, gid):
    b = ALPS_BATCH[gid]
    return f'{_alps_base}/{_alps_sub}/centraleurope_{var}_r1_alps_batch{b:02d}.dat'

vol_alps  = np.array([read_glogem_row(alps_dat('Volume', g), g) for g in GLACIER_IDS])
area_alps = np.array([read_glogem_row(alps_dat('Area',   g), g) for g in GLACIER_IDS])

I_2020 = 2020 - TRAN0
print(f"{'Glacier':<16}  {'ens_vol@2020':>13}  {'alps_vol@2020':>13}  {'ratio':>6}")
print('-' * 56)
for g in range(N_GL):
    v_e = vol_flow[g, I_2020]
    v_a = vol_alps[g, I_2020]
    ratio = v_a / v_e if v_e > 0 else float('nan')
    print(f'{GLACIER_NAMES[g]:<16}  {v_e:>13.4f}  {v_a:>13.4f}  {ratio:>6.2f}')

## 2. Ensemble relative-volume timeseries

One line per glacier, coloured by glacier area (log scale).  
Left: Δh parameterisation. Right: GloGEMflow.  
Y-axis: volume as % of the 1940 starting value.

In [ ]:
norm = mcolors.LogNorm(vmin=GLACIER_AREAS.min(), vmax=GLACIER_AREAS.max())
cmap = plt.cm.viridis

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)
titles = ['GloGEM Δh param.', 'GloGEMflow']
data_vols = [vol_dh, vol_flow]

for ax, title, vol in zip(axes, titles, data_vols):
    for g in range(N_GL):
        v0 = vol[g, 0]
        if v0 > 0:
            ax.plot(years, vol[g] / v0 * 100,
                    color=cmap(norm(GLACIER_AREAS[g])), lw=1.4, alpha=0.85)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Year')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

axes[0].set_ylabel('Volume (% of 1940)')

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, shrink=0.8, pad=0.02)
cbar.set_label('Glacier area (km$^2$)', fontsize=10)

fig.tight_layout()
plt.show()

## 3. Per-glacier area + volume panels

Each panel: area (dashed) and volume (solid), both normalised to 1940 value.  
Blue = GloGEM Δh param., orange = GloGEMflow.

In [ ]:
from matplotlib.lines import Line2D

NCOLS = 4
NROWS = (N_GL + NCOLS - 1) // NCOLS
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(16, NROWS * 3.2), sharex=True)
axes = axes.flatten()

legend_elements = [
    Line2D([0], [0], color=COL_DH,   lw=1.8,             label='Δh param. — volume'),
    Line2D([0], [0], color=COL_DH,   lw=1.4, ls='--',    label='Δh param. — area'),
    Line2D([0], [0], color=COL_FLOW, lw=1.8,             label='flow — volume'),
    Line2D([0], [0], color=COL_FLOW, lw=1.4, ls='--',    label='flow — area'),
]

for g in range(N_GL):
    ax = axes[g]
    a0_d = area_dh[g, 0] if area_dh[g, 0] > 0 else 1.0
    v0_d = vol_dh[g, 0]  if vol_dh[g, 0]  > 0 else 1.0
    a0_f = area_flow[g, 0] if area_flow[g, 0] > 0 else 1.0
    v0_f = vol_flow[g, 0]  if vol_flow[g, 0]  > 0 else 1.0

    ax.plot(years, area_dh[g] / a0_d * 100, color=COL_DH,   lw=1.4, ls='--', alpha=0.85)
    ax.plot(years, vol_dh[g]  / v0_d * 100, color=COL_DH,   lw=1.8,          alpha=0.85)
    ax.plot(years, area_flow[g] / a0_f * 100, color=COL_FLOW, lw=1.4, ls='--', alpha=0.85)
    ax.plot(years, vol_flow[g]  / v0_f * 100, color=COL_FLOW, lw=1.8,          alpha=0.85)

    ax.set_title(f'{GLACIER_NAMES[g]} ({GLACIER_AREAS[g]:.2f} km$^2$)', fontsize=9)
    ax.set_ylim(0, 105)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=4))
    if g % NCOLS == 0:
        ax.set_ylabel('% of 1940', fontsize=9)
    if g == 0:
        ax.legend(handles=legend_elements, fontsize=7, loc='lower left')

for ax in axes[N_GL:]:
    ax.set_visible(False)

fig.tight_layout()
plt.show()

## 4. 2100 volume-loss summary

Grouped bars: remaining volume at 2100 as % of 1940 value.  
Glaciers sorted by area (largest left).

In [ ]:
i_yr0  = 0
i_2100 = 2100 - TRAN0

sort_idx      = np.argsort(GLACIER_AREAS)[::-1]
names_sorted  = [GLACIER_NAMES[i] for i in sort_idx]
areas_sorted  = GLACIER_AREAS[sort_idx]

vfrac_dh   = np.array([
    vol_dh[i, i_2100] / vol_dh[i, i_yr0] * 100 if vol_dh[i, i_yr0] > 0 else 0
    for i in sort_idx
])
vfrac_flow  = np.array([
    vol_flow[i,  i_2100] / vol_flow[i,  i_yr0] * 100 if vol_flow[i,  i_yr0] > 0 else 0
    for i in sort_idx
])

print(f"{'Glacier':<16} {'Area (km2)':>10} {'Δh V%':>8} {'flow V%':>8}")
print('-' * 48)
for name, area, vd, vf in zip(names_sorted, areas_sorted, vfrac_dh, vfrac_flow):
    print(f'{name:<16} {area:>10.2f} {vd:>8.1f} {vf:>8.1f}')

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(N_GL, dtype=float)
w = 0.38
ax.bar(x - w/2, vfrac_dh,   w, color=COL_DH,   label='GloGEM Δh param.')
ax.bar(x + w/2, vfrac_flow,  w, color=COL_FLOW,  label='GloGEMflow')
ax.set_xticks(x)
ax.set_xticklabels(
    [f'{n}\n({a:.1f} km$^2$)' for n, a in zip(names_sorted, areas_sorted)],
    fontsize=8, rotation=30, ha='right'
)
ax.set_ylabel('Remaining volume (% of 1940 value)')
ax.set_title('Volume fraction remaining at 2100 (BCC-CSM2-MR / ssp126)', fontsize=12)
ax.set_ylim(0, 105)
ax.legend(loc='upper right', fontsize=10)
fig.tight_layout()
plt.show()

## 5. MB bias table (Hugonnet 2000–2019 period)

Mean annual MB in m w.e. yr⁻¹ for both models vs Hugonnet geodetic observations.

In [ ]:
print(f"{'Glacier':<16} {'Area':>7} {'Hugonnet':>10} {'Δh':>8} {'flow':>8} {'bias_dh':>8} {'bias_f':>8}")
print('-' * 72)
for g in range(N_GL):
    m_d = mb_dh[g,   i_hug0:i_hug1].mean()
    m_f = mb_flow[g, i_hug0:i_hug1].mean()
    print(
        f'{GLACIER_NAMES[g]:<16} {GLACIER_AREAS[g]:>7.2f}'
        f' {hug_mb[g]:>10.3f} {m_d:>8.3f} {m_f:>8.3f}'
        f' {m_d - hug_mb[g]:>+8.3f} {m_f - hug_mb[g]:>+8.3f}'
    )
print('\nAll values in m w.e. yr-1')

## 6. ensemble_flow vs alps_flow batch run — same 12 glaciers

Absolute volume (km³) for each glacier from three runs, all BCC-CSM2-MR / ssp126:

| Line | Source | What it tests |
|------|--------|---------------|
| **dashed blue** | Δh param. (ensemble) | reference Δh parameterisation |
| **solid orange** | ensemble_flow (dedicated run) | validated single-glacier spin-ups |
| **dotted green** | alps_flow (batch run) | batch spin-up — should match orange |

If the green line **jumps above** orange at the survey year, Phase B inflation is present in the batch spin-up for that glacier. The subtitle of each panel shows the 2020 volume for both flow runs and their ratio.

In [ ]:
COL_ALPS = '#1B5E20'  # dark green

NCOLS2 = 4
NROWS2 = (N_GL + NCOLS2 - 1) // NCOLS2
fig, axes = plt.subplots(NROWS2, NCOLS2, figsize=(16, NROWS2 * 3.4), sharex=True)
axes = axes.flatten()

legend_els = [
    plt.Line2D([0], [0], color=COL_DH,   lw=1.2, ls='--', label='Δh param.'),
    plt.Line2D([0], [0], color=COL_FLOW,  lw=1.8,          label='ensemble_flow'),
    plt.Line2D([0], [0], color=COL_ALPS,  lw=1.4, ls=':',  label='alps_flow (batch)'),
]

for g in range(N_GL):
    ax = axes[g]
    ax.plot(years, vol_dh[g],   color=COL_DH,   lw=1.2, ls='--', alpha=0.75)
    ax.plot(years, vol_flow[g], color=COL_FLOW,  lw=1.8,          alpha=0.90)
    ax.plot(years, vol_alps[g], color=COL_ALPS,  lw=1.4, ls=':',  alpha=0.90)

    ax.axvline(2020, color='0.5', lw=0.7, ls=':')
    ax.set_xlim(2000, 2100)
    ax.set_ylim(bottom=0)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=4))

    v_e = vol_flow[g, I_2020]
    v_a = vol_alps[g, I_2020]
    ratio = v_a / v_e if v_e > 0 else float('nan')
    ax.set_title(
        f'{GLACIER_NAMES[g]} ({GLACIER_AREAS[g]:.2f} km²)\n'
        f'ens={v_e:.3f}  alps={v_a:.3f} km³  ratio={ratio:.2f}×',
        fontsize=8
    )
    if g % NCOLS2 == 0:
        ax.set_ylabel('Volume (km³)', fontsize=9)
    if g == 0:
        ax.legend(handles=legend_els, fontsize=7, loc='upper right')

for ax in axes[N_GL:]:
    ax.set_visible(False)

fig.suptitle('ensemble_flow vs alps_flow batch run — same 12 glaciers (BCC-CSM2-MR / ssp126)',
             fontsize=11, y=1.01)
fig.tight_layout()
plt.show()